# Travel Reimbursement Approval Agent

This notebook evaluates the five supplied claims using an auditable tool-calling workflow. Financial decisions are deterministic; an LLM may optionally refine wording, but it cannot change amounts, decisions, or policy citations.

In [1]:
from __future__ import annotations
from collections import Counter
from copy import deepcopy
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from html import escape
from IPython.display import HTML, display
import json

D = Decimal
MONEY = D('0.01')
def money(value): return D(str(value)).quantize(MONEY, rounding=ROUND_HALF_UP)

POLICY = {
 'POL-CAT-01': 'Eligible: economy airfare, lodging, meals, ground transport, conference fees for documented business purpose.',
 'POL-CAT-02': 'Alcohol, minibar, spa, gym, entertainment, shopping, gifts, fines, fees, and personal expenses are ineligible.',
 'POL-PD-01': 'Meals are capped at $75 per day.',
 'POL-PD-02': 'Lodging is capped at $200 per night.',
 'POL-PD-03': 'Ground transport is capped at $50 per day.',
 'POL-AIR-01': 'Only economy airfare is reimbursable; business/first class requires manual review.',
 'POL-RCT-01': 'Items above $25 require itemized receipts; airfare and lodging always require receipts.',
 'POL-RCT-02': 'Missing required receipts route the claim to manual review.',
 'POL-APR-01': 'Compliant reimbursable total up to $500 may be auto-approved.',
 'POL-APR-02': 'Compliant reimbursable total above $500 through $2,000 is approvable in the manager tier.',
 'POL-APR-03': 'Reimbursable total above $2,000 requires director/manual review.',
 'POL-TIME-01': 'Claims submitted more than 30 days after the expense date require manual review.'
}
ELIGIBLE = {'airfare','lodging','meals','ground_transport','conference_fees'}
INELIGIBLE = {'alcohol','minibar','spa','gym','entertainment','shopping','gifts','traffic_fines','penalties','late_fees','personal'}
DECISIONS = {'APPROVE','PARTIAL_APPROVE','REJECT','MANUAL_REVIEW'}

In [2]:
CLAIMS = [
 {'claim_id':'CLM-001','employee':'A. Rivera','business_purpose':'Attend 2-day industry conference','trip_start':'2026-06-10','trip_end':'2026-06-12','submitted':'2026-06-20','items':[
   {'category':'airfare','description':'Round-trip economy airfare','amount':420,'receipt':True},
   {'category':'lodging','description':'Hotel, 2 nights @ $180','amount':360,'receipt':True,'units':2},
   {'category':'meals','description':'Meals, 3 days @ ~$60/day','amount':180,'receipt':True,'units':3},
   {'category':'conference_fees','description':'Conference registration','amount':150,'receipt':True}]},
 {'claim_id':'CLM-002','employee':'B. Osei','business_purpose':'Weekend hotel stay','trip_start':'2026-06-14','trip_end':'2026-06-15','submitted':'2026-06-25','items':[
   {'category':'spa','description':'Hotel spa package','amount':300,'receipt':True},
   {'category':'minibar','description':'In-room minibar','amount':80,'receipt':True}]},
 {'claim_id':'CLM-003','employee':'C. Nakamura','business_purpose':'Client site visit','trip_start':'2026-06-08','trip_end':'2026-06-10','submitted':'2026-06-22','items':[
   {'category':'airfare','description':'Round-trip economy airfare','amount':300,'receipt':True},
   {'category':'lodging','description':'Hotel, 2 nights @ $250','amount':500,'receipt':True,'units':2},
   {'category':'meals','description':'Meals, 2 days @ $70/day','amount':140,'receipt':True,'units':2}]},
 {'claim_id':'CLM-004','employee':'D. Fischer','business_purpose':'International vendor negotiation','trip_start':'2026-06-16','trip_end':'2026-06-18','submitted':'2026-06-28','items':[
   {'category':'airfare','description':'Business-class international airfare','amount':2400,'receipt':True,'travel_class':'business'},
   {'category':'lodging','description':'Hotel, 3 nights','amount':600,'receipt':False,'units':3}]},
 {'claim_id':'CLM-005','employee':'E. Haddad','business_purpose':'Client dinner / business development','trip_start':'2026-06-11','trip_end':'2026-06-11','submitted':'2026-06-24','items':[
   {'category':'meals','description':'Client dinner for 4 (business development)','amount':220,'receipt':False,'units':1}]}
]
for claim in CLAIMS: claim['total_claimed'] = sum(money(i['amount']) for i in claim['items'])
assert [str(c['total_claimed']) for c in CLAIMS] == ['1110.00','380.00','940.00','3000.00','220.00']

## Agent Tools
Each tool returns a machine-readable observation and the policy IDs it used. The orchestrator decides which tools apply and records a complete audit trail.

In [3]:
def policy_lookup(claim):
    refs = {'POL-CAT-01','POL-CAT-02','POL-RCT-01','POL-RCT-02','POL-TIME-01'}
    for item in claim['items']:
        refs |= {'meals':{'POL-PD-01'},'lodging':{'POL-PD-02'},'ground_transport':{'POL-PD-03'},'airfare':{'POL-AIR-01'}}.get(item['category'],set())
    return {'policy_refs': sorted(refs), 'context': {r:POLICY[r] for r in sorted(refs)}}

def category_check(claim):
    results=[]
    for i in claim['items']:
        status = 'eligible' if i['category'] in ELIGIBLE else 'ineligible' if i['category'] in INELIGIBLE else 'ambiguous'
        results.append({'category':i['category'],'status':status,'amount':money(i['amount'])})
    return {'items':results,'policy_refs':['POL-CAT-01','POL-CAT-02']}

def receipt_completeness_check(claim):
    missing=[]
    for i in claim['items']:
        required = i['category'] in {'airfare','lodging'} or money(i['amount']) > D('25')
        if required and not i['receipt']: missing.append(f"Itemized receipt: {i['description']} (${money(i['amount'])})")
    return {'missing_docs':missing,'manual_review':bool(missing),'policy_refs':['POL-RCT-01'] + (['POL-RCT-02'] if missing else [])}

def limit_check(claim):
    caps={'meals':D('75'),'lodging':D('200'),'ground_transport':D('50')}
    approved=deducted=D('0'); details=[]; exceptions=[]
    for i in claim['items']:
        amount=money(i['amount']); category=i['category']
        if category in INELIGIBLE:
            allowed=D('0'); reason='ineligible category'
        elif category=='airfare' and i.get('travel_class','economy').lower()!='economy':
            allowed=amount; reason='airfare class exception'; exceptions.append(i['description'])
        elif category in caps:
            maximum=caps[category]*D(i.get('units',1)); allowed=min(amount,maximum); reason=f'cap ${caps[category]} x {i.get("units",1)}'
        elif category in ELIGIBLE:
            allowed=amount; reason='eligible category'
        else:
            allowed=amount; reason='unknown category'; exceptions.append(i['description'])
        deduction=amount-allowed; approved+=allowed; deducted+=deduction
        details.append({'category':category,'claimed':amount,'provisionally_approved':allowed,'deducted':deduction,'reason':reason})
    refs=['POL-PD-01','POL-PD-02','POL-PD-03','POL-AIR-01','POL-CAT-02']
    return {'approved':money(approved),'deducted':money(deducted),'details':details,'exceptions':exceptions,'policy_refs':refs}

def timeliness_check(claim):
    days=(date.fromisoformat(claim['submitted'])-date.fromisoformat(claim['trip_end'])).days
    return {'days_after_trip':days,'late':days>30,'policy_refs':['POL-TIME-01']}

def duplicate_detector(claim, all_claims):
    signatures=[]
    for other in all_claims:
        if other['claim_id']==claim['claim_id']: continue
        for item in claim['items']:
            for candidate in other['items']:
                if item['category']==candidate['category'] and money(item['amount'])==money(candidate['amount']) and claim['trip_start']==other['trip_start']:
                    signatures.append(other['claim_id'])
    return {'possible_duplicates':sorted(set(signatures)),'manual_review':bool(signatures),'policy_refs':[]}

def approval_threshold_check(reimbursable):
    if reimbursable>D('2000'): return {'tier':'director','manual_review':True,'policy_refs':['POL-APR-03']}
    if reimbursable>D('500'): return {'tier':'manager','manual_review':False,'policy_refs':['POL-APR-02']}
    return {'tier':'auto','manual_review':False,'policy_refs':['POL-APR-01']}

## Agent Orchestration and Guardrails
Precedence is conservative: missing documents, exceptions, ambiguity, duplicates, lateness, or director-level totals force `MANUAL_REVIEW`. A claim is rejected only when nothing is reimbursable and all items are explicitly ineligible.

In [4]:
REQUIRED_FIELDS=['claim_id','decision','approved_amount','deducted_amount','missing_docs','policy_refs','confidence','explanation','tools_used']
def output_validator(result):
    assert list(result)==REQUIRED_FIELDS, f'Output fields/order invalid: {list(result)}'
    assert result['decision'] in DECISIONS
    assert isinstance(result['approved_amount'],float) and isinstance(result['deducted_amount'],float)
    assert result['approved_amount']>=0 and result['deducted_amount']>=0
    assert 0<=result['confidence']<=1
    assert all(r in POLICY for r in result['policy_refs'])
    return True

def evaluate_claim(claim, all_claims):
    tools_used=[]; observations={}
    def call(name, fn, *args):
        tools_used.append(name); observations[name]=fn(*args); return observations[name]
    context=call('policy_lookup',policy_lookup,claim)
    categories=call('category_check',category_check,claim)
    receipts=call('receipt_completeness_check',receipt_completeness_check,claim)
    limits=call('limit_check',limit_check,claim)
    timing=call('timeliness_check',timeliness_check,claim)
    duplicate=call('duplicate_detector',duplicate_detector,claim,all_claims)
    threshold=call('approval_threshold_check',approval_threshold_check,limits['approved'])
    statuses=[x['status'] for x in categories['items']]
    manual_reasons=[]
    if receipts['manual_review']: manual_reasons.append('required receipt missing')
    if limits['exceptions']: manual_reasons.append('policy exception or ambiguity')
    if timing['late']: manual_reasons.append('submitted after 30-day window')
    if duplicate['manual_review']: manual_reasons.append('possible duplicate')
    if threshold['manual_review']: manual_reasons.append('director approval required')
    if 'ambiguous' in statuses: manual_reasons.append('unknown category')
    if manual_reasons: decision='MANUAL_REVIEW'
    elif limits['approved']==0 and set(statuses)=={'ineligible'}: decision='REJECT'
    elif limits['deducted']>0: decision='PARTIAL_APPROVE'
    else: decision='APPROVE'
    refs=set()
    for value in observations.values(): refs.update(value.get('policy_refs',[]))
    if decision=='MANUAL_REVIEW': explanation=f"Manual review required: {', '.join(manual_reasons)}. Provisional policy calculation allows ${limits['approved']:.2f} and deducts ${limits['deducted']:.2f}."
    elif decision=='REJECT': explanation=f"All claimed items are explicitly ineligible; the full ${claim['total_claimed']:.2f} is deducted."
    elif decision=='PARTIAL_APPROVE': explanation=f"The claim is valid, but policy caps reduce reimbursement to ${limits['approved']:.2f}; excess of ${limits['deducted']:.2f} is deducted."
    else: explanation=f"All items are eligible, documented, timely, and within policy limits; ${limits['approved']:.2f} is approvable in the {threshold['tier']} tier."
    confidence=0.99 if decision in {'APPROVE','REJECT'} else 0.96 if decision=='PARTIAL_APPROVE' else 0.88
    result={'claim_id':claim['claim_id'],'decision':decision,'approved_amount':float(limits['approved']),'deducted_amount':float(limits['deducted']),'missing_docs':receipts['missing_docs'],'policy_refs':sorted(refs),'confidence':confidence,'explanation':explanation,'tools_used':tools_used+['output_validator']}
    output_validator(result)
    return result, observations

RESULTS=[]; AUDIT_TRAIL={}
for claim in CLAIMS:
    result,audit=evaluate_claim(claim,CLAIMS); RESULTS.append(result); AUDIT_TRAIL[claim['claim_id']]=audit
assert [r['decision'] for r in RESULTS]==['APPROVE','REJECT','PARTIAL_APPROVE','MANUAL_REVIEW','MANUAL_REVIEW']
print('Validated',len(RESULTS),'claim recommendations.')

Validated 5 claim recommendations.


## Audit TrailThe following compact view demonstrates context retrieval and tool calls. Full observations remain available in `AUDIT_TRAIL`.

In [5]:
for result in RESULTS:
    print(f"{result['claim_id']}: {result['decision']} | tools={len(result['tools_used'])} | refs={', '.join(result['policy_refs'])}")

CLM-001: APPROVE | tools=8 | refs=POL-AIR-01, POL-APR-02, POL-CAT-01, POL-CAT-02, POL-PD-01, POL-PD-02, POL-PD-03, POL-RCT-01, POL-RCT-02, POL-TIME-01
CLM-002: REJECT | tools=8 | refs=POL-AIR-01, POL-APR-01, POL-CAT-01, POL-CAT-02, POL-PD-01, POL-PD-02, POL-PD-03, POL-RCT-01, POL-RCT-02, POL-TIME-01
CLM-003: PARTIAL_APPROVE | tools=8 | refs=POL-AIR-01, POL-APR-02, POL-CAT-01, POL-CAT-02, POL-PD-01, POL-PD-02, POL-PD-03, POL-RCT-01, POL-RCT-02, POL-TIME-01
CLM-004: MANUAL_REVIEW | tools=8 | refs=POL-AIR-01, POL-APR-03, POL-CAT-01, POL-CAT-02, POL-PD-01, POL-PD-02, POL-PD-03, POL-RCT-01, POL-RCT-02, POL-TIME-01
CLM-005: MANUAL_REVIEW | tools=8 | refs=POL-AIR-01, POL-APR-01, POL-CAT-01, POL-CAT-02, POL-PD-01, POL-PD-02, POL-PD-03, POL-RCT-01, POL-RCT-02, POL-TIME-01


## Dashboard

In [6]:
counts=Counter(r['decision'] for r in RESULTS); approved=sum(r['approved_amount'] for r in RESULTS); deducted=sum(r['deducted_amount'] for r in RESULTS)
colors={'APPROVE':'#16a34a','PARTIAL_APPROVE':'#d97706','REJECT':'#dc2626','MANUAL_REVIEW':'#7c3aed'}
cards=''.join(f"<div class='card'><b>{escape(k.replace('_',' '))}</b><strong style='color:{colors[k]}'>{counts.get(k,0)}</strong></div>" for k in ['APPROVE','PARTIAL_APPROVE','REJECT','MANUAL_REVIEW'])
rows=''.join(f"<tr><td>{r['claim_id']}</td><td style='color:{colors[r['decision']]}'>{r['decision']}</td><td>${r['approved_amount']:,.2f}</td><td>${r['deducted_amount']:,.2f}</td><td>{r['confidence']:.0%}</td></tr>" for r in RESULTS)
display(HTML(f"""<style>.dash{{font-family:system-ui;background:#f8fafc;padding:20px;border-radius:16px}}.cards{{display:flex;gap:10px;flex-wrap:wrap}}.card{{background:white;padding:14px;border-radius:10px;min-width:150px;box-shadow:0 1px 5px #0002}}.card strong{{display:block;font-size:28px}}table{{width:100%;border-collapse:collapse;margin-top:16px;background:white}}th,td{{padding:9px;border-bottom:1px solid #ddd;text-align:left}}</style><div class='dash'><h2>Travel Reimbursement Results</h2><div class='cards'>{cards}<div class='card'><b>Provisionally approved</b><strong>${approved:,.2f}</strong></div><div class='card'><b>Policy deductions</b><strong>${deducted:,.2f}</strong></div></div><table><tr><th>Claim</th><th>Decision</th><th>Approved</th><th>Deducted</th><th>Confidence</th></tr>{rows}</table></div>"""))

Claim,Decision,Approved,Deducted,Confidence
CLM-001,APPROVE,"$1,110.00",$0.00,99%
CLM-002,REJECT,$0.00,$380.00,99%
CLM-003,PARTIAL_APPROVE,$840.00,$100.00,96%
CLM-004,MANUAL_REVIEW,"$3,000.00",$0.00,88%
CLM-005,MANUAL_REVIEW,$75.00,$145.00,88%


## Design Notes & Reasoning

- **Grounding:** Stable `POL-*` rules are represented as an explicit policy store. `policy_lookup` retrieves only rules relevant to categories present in a claim.
- **Agentic workflow:** The orchestrator calls seven domain tools, combines their observations, applies decision precedence, and invokes an output validator. `tools_used` and `AUDIT_TRAIL` make this observable.
- **Reliability trade-off:** An LLM is intentionally not allowed to perform arithmetic or make the final financial decision. A production LLM could rewrite explanations or resolve natural-language intake, but validated rule results remain authoritative. This makes the notebook free, offline, reproducible, and resistant to hallucinated policy.
- **Manual review:** CLM-004 has business-class airfare, a missing lodging receipt, and exceeds $2,000. CLM-005 has a required missing receipt. Both are routed to a person rather than forcing approval/rejection. Their amounts are provisional pending review.
- **Dates:** Timeliness uses trip end as the latest expense date because line-item dates are not supplied.
- **Per-diem:** Provided day/night counts are treated as authoritative. The client dinner is one meal-day, so its provisional cap is $75.
- **Known gaps:** No OCR, receipt image authenticity check, currency conversion, employee history, or persistent duplicate index is included.
- **Next improvements:** Add receipt OCR, schema-constrained LLM extraction for unstructured claims, a persistent duplicate store, human-review actions, policy versioning, and evaluation against labelled historical cases.

## Final Structured ResultsThe final code cell emits only the required JSON array, with exactly the specified fields.

In [7]:
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-AIR-01",
      "POL-APR-02",
      "POL-CAT-01",
      "POL-CAT-02",
      "POL-PD-01",
      "POL-PD-02",
      "POL-PD-03",
      "POL-RCT-01",
      "POL-RCT-02",
      "POL-TIME-01"
    ],
    "confidence": 0.99,
    "explanation": "All items are eligible, documented, timely, and within policy limits; $1110.00 is approvable in the manager tier.",
    "tools_used": [
      "policy_lookup",
      "category_check",
      "receipt_completeness_check",
      "limit_check",
      "timeliness_check",
      "duplicate_detector",
      "approval_threshold_check",
      "output_validator"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-AIR-01",
      "POL-APR-01",
      "POL-CAT-01",
    